# Train DQN Agent on Ms. Pac-Man

This notebook demonstrates training a Deep Q-Network (DQN) agent to play Ms. Pac-Man.

## 1. Setup

In [ ]:
# For Google Colab
# !git clone https://github.com/SABRYOLA/pacman.git
# %cd pacman
# !pip install -r requirements.txt

In [ ]:
import torch
import yaml
import sys
sys.path.append('..')

from src.agents import DQNAgent
from src.environment import create_env
from src.training import DQNTrainer
from src.utils import Logger, CheckpointManager
from src.networks import get_device

## 2. Configuration

In [ ]:
# Load config
with open('../configs/dqn_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Reduce steps for notebook demo
config['total_timesteps'] = 100000  # 100K for quick demo
config['learning_starts'] = 10000

print("DQN Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Create Environment and Agent

In [ ]:
# Device
device = get_device(config['device'])
print(f"Using device: {device}")

# Create environment
env = create_env(config['env_name'], seed=config['seed'])
eval_env = create_env(config['env_name'], seed=config['seed'] + 1000)

n_actions = env.action_space.n
print(f"Environment: {config['env_name']}")
print(f"Number of actions: {n_actions}")

In [ ]:
# Create agent
agent = DQNAgent(
    n_actions=n_actions,
    learning_rate=config['learning_rate'],
    gamma=config['gamma'],
    buffer_size=config['buffer_size'],
    batch_size=config['batch_size'],
    target_update_interval=config['target_update_interval'],
    exploration_initial_eps=config['exploration_initial_eps'],
    exploration_final_eps=config['exploration_final_eps'],
    exploration_fraction=config['exploration_fraction'],
    device=device
)

print(f"\nAgent created with {sum(p.numel() for p in agent.q_network.parameters()):,} parameters")

## 4. Create Logger and Trainer

In [ ]:
# Create logger and checkpoint manager
logger = Logger('../logs/dqn_notebook', 'DQN')
checkpoint_manager = CheckpointManager('../models', 'DQN')

# Create trainer
trainer = DQNTrainer(
    agent=agent,
    env=env,
    logger=logger,
    checkpoint_manager=checkpoint_manager,
    total_timesteps=config['total_timesteps'],
    learning_starts=config['learning_starts'],
    train_freq=config['train_freq'],
    eval_env=eval_env,
    eval_freq=config['eval_freq'],
    eval_episodes=config['eval_episodes'],
    save_freq=config['save_freq'],
    log_interval=config['log_interval']
)

print("Trainer ready!")

## 5. Train!

In [ ]:
# Start training
print(f"Starting training for {config['total_timesteps']:,} steps...\n")
trainer.train()
print("\nTraining complete!")

## 6. View Results

In [ ]:
# Get training statistics
stats = logger.get_stats()

print("\nTraining Statistics:")
print(f"  Total steps: {stats['total_steps']:,}")
print(f"  Episodes: {stats['num_episodes']}")
print(f"  Mean reward (last 100): {stats.get('mean_reward', 0):.2f}")
print(f"  Max reward: {stats.get('max_reward', 0):.2f}")
print(f"  Final epsilon: {agent.epsilon:.4f}")

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir ../logs/

## 7. Test Trained Agent

In [ ]:
# Evaluate agent
import numpy as np

test_rewards = []
test_env = create_env(config['env_name'], seed=999)

for episode in range(10):
    obs, _ = test_env.reset()
    episode_reward = 0
    
    while True:
        action = agent.select_action(obs, training=False)
        obs, reward, terminated, truncated, _ = test_env.step(action)
        episode_reward += reward
        
        if terminated or truncated:
            break
    
    test_rewards.append(episode_reward)
    print(f"Episode {episode+1}: {episode_reward:.0f}")

print(f"\nTest Performance: {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")
test_env.close()

## 8. Save Model

In [ ]:
# Save final model
agent.save('../models/dqn_notebook_final.pt')
print("Model saved!")

## Next Steps

- Try training for longer (1M+ steps)
- Experiment with hyperparameters
- Compare with PPO and A2C (notebooks 03-04)
- Visualize learned Q-values

In [ ]:
# Cleanup
env.close()
eval_env.close()
logger.close()